In [1]:
import pandas as pd
import matplotlib.pyplot as plt


df = pd.read_csv('airports-extended-clean.csv', sep=';')

df.head()

,Airport ID,Name,City,Country,IATA,ICAO,Latitude,Longitude,Altitude,Timezone,DST,Tz,Type,Source
0,1,Goroka Airport,Goroka,Papua New Guinea,GKA,AYGA,"-6,081689835","145,3919983",5282,10,U,Pacific/Port_Moresby,airport,OurAirports
1,2,Madang Airport,Madang,Papua New Guinea,MAG,AYMD,"-5,207079887","145,7890015",20,10,U,Pacific/Port_Moresby,airport,OurAirports
2,3,Mount Hagen Kagamuga Airport,Mount Hagen,Papua New Guinea,HGU,AYMH,"-5,826789856","144,2960052",5388,10,U,Pacific/Port_Moresby,airport,OurAirports
3,4,Nadzab Airport,Nadzab,Papua New Guinea,LAE,AYNZ,"-6,569803","146,725977",239,10,U,Pacific/Port_Moresby,airport,OurAirports
4,5,Port Moresby Jacksons International Airport,Port Moresby,Papua New Guinea,POM,AYPY,"-9,443380356","147,2200012",146,10,U,Pacific/Port_Moresby,airport,OurAirports


In [2]:
df_nan = df[df['City'].isna()]

print(df_nan)


       Airport ID                                Name City       Country IATA  \
10351       11794  Minsk Mazowiecki Military Air Base  NaN        Poland   \N   
10352       11795            Powidz Military Air Base  NaN        Poland   \N   
10457       11900            Dawadmi Domestic Airport  NaN  Saudi Arabia   \N   
10458       11901                King Khaled Air Base  NaN  Saudi Arabia  KMX   
10478       11921                   Asahikawa Airport  NaN         Japan   \N   
10480       11923                  Utsunomiya Airport  NaN         Japan  QUT   
10481       11924                    Jungwon Air Base  NaN   South Korea   \N   
10484       11927                      Bislig Airport  NaN   Philippines  BPH   
10485       11928               Mati National Airport  NaN   Philippines  MXI   
10504       11947               Metropolitano Airport  NaN     Venezuela   \N   
10511       11954                 Belaya Gora Airport  NaN        Russia   \N   
10541       11984           

In [3]:
totaal_rijen = len(df)

print(totaal_rijen)

print(df.isna().sum()/len(df) * 100)






10668
Airport ID    0.000000
Name          0.000000
City          0.440570
Country       0.000000
IATA          0.018748
ICAO          0.009374
Latitude      0.000000
Longitude     0.000000
Altitude      0.000000
Timezone      0.000000
DST           0.000000
Tz            0.000000
Type          0.000000
Source        0.000000
dtype: float64


In [4]:
print(df_nan['Country'].value_counts())

Country
Australia       30
India            4
Poland           2
Saudi Arabia     2
Japan            2
Philippines      2
Russia           2
South Korea      1
Venezuela        1
Mongolia         1
Name: count, dtype: int64


In [5]:
#Filter op land en stad waar geen data is. 

Australia_nan_cities = df[(df['Country'] == 'Australia') & (df['City'].isna())]

print(Australia_nan_cities)

print(f"Aantal Australische vliegvelden zonder stad: {len(Australia_nan_cities)}")


       Airport ID                   Name City    Country IATA  ICAO  \
10569       12012         Ararat Airport  NaN  Australia  ARY  YARA   
10570       12013        Benalla Airport  NaN  Australia  BLN  YBLA   
10571       12014      Balranald Airport  NaN  Australia  BZD  YBRN   
10572       12015     Brewarrina Airport  NaN  Australia  BWQ  YBRW   
10573       12016          Cleve Airport  NaN  Australia  CVC  YCEE   
10574       12017         Corowa Airport  NaN  Australia  CWW  YCOR   
10575       12018       Corryong Airport  NaN  Australia  CYG  YCRG   
10576       12019    Cootamundra Airport  NaN  Australia  CMD  YCTM   
10577       12020    Dirranbandi Airport  NaN  Australia  DRN  YDBI   
10579       12022         Dysart Airport  NaN  Australia  DYA  YDYS   
10580       12023         Echuca Airport  NaN  Australia  ECH  YECH   
10582       12025       Gunnedah Airport  NaN  Australia  GUH  YGDH   
10583       12026            Hay Airport  NaN  Australia  HXX  YHAY   
10584 

In [6]:
# 1. Maak een lijst van vliegveldnamen met een bekende stad
bekende_steden = df[df['City'].notna()].set_index('Name')['City'].to_dict()

# 2. Filter op rijen waar City NaN is
mist_stad = df[df['City'].isna()]

# 3. Kijk welke van deze namen wél voorkomen in onze lijst van bekende steden
herstelbaar = mist_stad[mist_stad['Name'].isin(bekende_steden.keys())]

print("Deze regels hebben een NaN, maar de stad is elders bekend:")
print(herstelbaar[['Name', 'Country']])

Deze regels hebben een NaN, maar de stad is elders bekend:
                           Name       Country
10457  Dawadmi Domestic Airport  Saudi Arabia
10478         Asahikawa Airport         Japan


In [7]:
# We groeperen op 'Name' en vullen de NaN waarden in met de eerste waarde die hij tegenkomt voor die naam
df['City'] = df.groupby('Name')['City'].transform(lambda x: x.fillna(method='ffill').fillna(method='bfill'))

C:\Users\sander\AppData\Local\Temp\ipykernel_22896\375246296.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['City'] = df.groupby('Name')['City'].transform(lambda x: x.fillna(method='ffill').fillna(method='bfill'))
C:\Users\sander\AppData\Local\Temp\ipykernel_22896\375246296.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['City'] = df.groupby('Name')['City'].transform(lambda x: x.fillna(method='ffill').fillna(method='bfill'))


In [8]:
# Check via Airport ID
id_met_stad = df[df['City'].notna()].set_index('Airport ID')['City'].to_dict()
herstelbaar_via_id = df[df['City'].isna() & df['Airport ID'].isin(id_met_stad.keys())]

# Check via IATA (alleen voor rijen waar IATA niet \N of NaN is)
iata_met_stad = df[df['City'].notna() & (df['IATA'] != '\\N')].set_index('IATA')['City'].to_dict()
herstelbaar_via_iata = df[df['City'].isna() & df['IATA'].isin(iata_met_stad.keys())]

print(f"Aantal te herstellen via ID: {len(herstelbaar_via_id)}")
print(f"Aantal te herstellen via IATA: {len(herstelbaar_via_iata)}")

Aantal te herstellen via ID: 0
Aantal te herstellen via IATA: 0


In [9]:
# Verwijder alle rijen waar 'City' nog steeds NaN is
df = df.dropna(subset=['City'])

# Of, als je de wijziging direct in de huidige variabele wilt doorvoeren:
# df.dropna(subset=['City'], inplace=True)

# Controleer hoeveel rijen er over zijn
print(f"Aantal rijen na opschonen: {len(df)}")

Aantal rijen na opschonen: 10623


In [10]:
print(df['City'].isna().sum())

0


In [11]:
df_airports = df

df_flights = pd.read_csv('schedule_airport.csv')

In [12]:
# Tel hoeveel vluchten er vertrekken per vliegveld

flight_counts = df_flights.groupby('FLT').size().reset_index(name='FlightCount')

print(flight_counts)

         FLT  FlightCount
0      2L100            2
1      2L102           48
2      2L103           47
3      2L104            1
4     2L1040            1
...      ...          ...
2161  ZT2461            1
2162  ZT3401            1
2163  ZT3402            1
2164  ZT4361            1
2165  ZT4362            1

[2166 rows x 2 columns]


In [13]:
# We koppelen 'Origin' uit de tellingen aan 'IATA' uit de vliegveldlijst
df_merged = pd.merge(df_airports, flight_counts, left_on='IATA', right_on='FLT', how='left')

# Vliegvelden zonder vluchten krijgen een NaN bij FlightCount, die maken we 0
df_merged['FlightCount'] = df_merged['FlightCount'].fillna(0)

In [14]:
import plotly.express as px

fig = px.scatter_geo(df_merged, 
                     lat='Latitude', 
                     lon='Longitude', 
                     size='FlightCount', # De grootte van de bubbel
                     hover_name='Name',  # De tekst als je eroverheen zweeft
                     title='Aantal vluchten per luchthaven')
fig.show()

In [27]:
import pandas as pd
import plotly.express as px

# 1. Laad de vliegvelden (let op de puntkomma en de komma als decimaal)
df_airports = pd.read_csv('airports-extended-clean.csv', sep=';', decimal=',')

# 2. Laad de vluchten
df_schedule = pd.read_csv('schedule_airport.csv')

# 3. Tel het aantal vluchten per vliegveld (kolom 'Org/Des')
flight_counts = df_schedule.groupby('Org/Des').size().reset_index(name='FLT')

# 4. Voeg de data samen
# We koppelen 'ICAO' uit het vliegveld-bestand aan 'Org/Des' uit de tellingen
df_map = pd.merge(df_airports, flight_counts, left_on='ICAO', right_on='Org/Des', how='inner')

# 5. Maak het bubbeldiagram
fig = px.scatter_geo(
    df_map,
    lat='Latitude',
    lon='Longitude',
    size='FLT',               # Grootte op basis van aantal vluchten
    hover_name='Name',        # Toon naam van vliegveld bij hover
    color='FLT',              # Kleur op basis van drukte
    projection="natural earth",
    title='Drukte op luchthavens op basis van vliegschema'
)

fig.show()